# Fundamentos y metricas para LLMs

En los notebooks anteriores **construimos** sistemas con LLMs (RAG con LangChain y LangGraph).
A partir de ahora aprenderemos a **medirlos**.

### Objetivos de esta clase
- ✅ Entender por qué evaluar texto generado es difícil
- ✅ Implementar métricas léxicas: Exact Match, F1, BLEU, ROUGE
- ✅ Descubrir sus limitaciones con ejemplos reales
- ✅ Usar métricas semánticas basadas en embeddings
- ✅ Medir la **perplexity** de un modelo de lenguaje y entender qué significa
- ✅ Evaluar un LLM sobre un benchmark real (MMLU)
- ✅ Construir un **golden dataset** propio
- ✅ Generar datos de evaluación sintéticos con un LLM

### La pregunta central de esta clase

> Si un LLM responde *"París es la capital de Francia"* y otro responde
> *"La capital francesa es París"*... ¿cómo le decimos a una computadora
> que **ambas respuestas son correctas**?


## Instalación de Dependencias

In [ ]:
!pip install transformers torch sentence-transformers -q

In [ ]:
!pip install evaluate rouge_score sacrebleu datasets -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.5 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings('ignore')

print("✅ Ready to import")

✅ Ready to import


## Importaciones

In [ ]:
import numpy as np
import torch
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from sentence_transformers import SentenceTransformer
import evaluate

print("✅ Imports successful")
print(f"🔧 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

✅ Imports successful
🔧 Device: GPU


## Setup: LLM

Usamos el mismo modelo del notebook anterior (`flan-t5-base`) para mantener continuidad.
Este será nuestro **sujeto de evaluación**: el modelo cuyas respuestas vamos a medir.

In [ ]:
print("🔄 Loading LLM...")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=100,
    device=0 if torch.cuda.is_available() else -1
)

print("✅ LLM ready")

🔄 Loading LLM...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'Deep

✅ LLM ready


---
# PARTE 1: ¿Por qué es difícil evaluar un LLM?

En Machine Learning clásico, evaluar es "fácil":
- **Clasificación** → ¿acertó la clase? → accuracy, precision, recall
- **Regresión** → ¿qué tan lejos quedó del valor real? → MSE, MAE

Pero en generación de texto **no existe una única respuesta correcta**.

Veámoslo con un ejemplo:

In [ ]:
question = "What is the capital of France?"

# Tres respuestas, TODAS correctas
answers = [
    "Paris",
    "The capital of France is Paris.",
    "Paris is the capital and largest city of France, located on the Seine river."
]

# Y una respuesta INCORRECTA pero que "suena" similar
wrong_answer = "The capital of France is Lyon."

print(f"❓ Question: {question}\n")
for i, a in enumerate(answers, 1):
    print(f"✅ Answer {i}: {a}")
print(f"\n❌ Wrong:    {wrong_answer}")

❓ Question: What is the capital of France?

✅ Answer 1: Paris
✅ Answer 2: The capital of France is Paris.
✅ Answer 3: Paris is the capital and largest city of France, located on the Seine river.

❌ Wrong:    The capital of France is Lyon.


**El problema**: las 3 primeras respuestas son correctas pero tienen longitudes,
palabras y estructuras totalmente distintas. La cuarta comparte casi todas las palabras
con la respuesta 2... pero es **falsa**.

Cualquier métrica que usemos debe lidiar con esto. Empecemos por las más simples.

---
# PARTE 2: Métricas Léxicas (comparación de palabras)

## 2.1 Exact Match (EM)

La métrica más estricta: ¿la respuesta es **idéntica** a la referencia?

Se usa mucho en QA extractivo (ej. SQuAD), normalizando texto primero
(minúsculas, sin puntuación, sin artículos).

### 📐 Fundamento matemático

Sea $\hat{y}_i$ la predicción del modelo e $y_i$ la respuesta de referencia para el ejemplo $i$.
Sobre un dataset de $N$ ejemplos:

$$\text{EM} = \frac{1}{N}\sum_{i=1}^{N} \mathbb{1}\big[\,\text{norm}(\hat{y}_i) = \text{norm}(y_i)\,\big]$$

donde:
- $\mathbb{1}[\cdot]$ es la **función indicadora**: vale $1$ si la condición se cumple y $0$ si no
- $\text{norm}(\cdot)$ es una función de **normalización** de texto (minúsculas, sin puntuación, sin artículos, espacios colapsados)

**Propiedades**:
- Es una métrica **binaria por ejemplo** ($0$ o $1$) y un **promedio** a nivel de dataset ($\in [0,1]$)
- Es el caso extremo de exigencia: equivale a un accuracy donde la "clase" es la cadena de texto completa
- La normalización define qué diferencias "perdonamos" — cambiarla cambia la métrica


In [ ]:
import re
import string

def normalize_text(text):
    """Normalización estándar estilo SQuAD"""
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)   # quitar artículos
    text = " ".join(text.split())                   # normalizar espacios
    return text

def exact_match(prediction, reference):
    return int(normalize_text(prediction) == normalize_text(reference))

reference = "Paris"

print(f"Reference: '{reference}'\n")
for a in answers:
    print(f"EM = {exact_match(a, reference)}  ←  '{a}'")

Reference: 'Paris'

EM = 1  ←  'Paris'
EM = 0  ←  'The capital of France is Paris.'
EM = 0  ←  'Paris is the capital and largest city of France, located on the Seine river.'


**Resultado**: solo la respuesta más corta obtiene EM=1. Las otras dos respuestas
correctas obtienen **0**, igual que si hubieran dicho una barbaridad.

👉 EM es útil cuando la respuesta esperada es corta y cerrada (fechas, nombres, números),
pero castiga injustamente respuestas correctas más elaboradas.

## 2.2 Token F1

Más flexible: mide el **solapamiento de palabras** entre predicción y referencia.

- **Precision**: ¿qué fracción de las palabras de la predicción están en la referencia?
- **Recall**: ¿qué fracción de las palabras de la referencia aparecen en la predicción?
- **F1**: media armónica de ambas

### 📐 Fundamento matemático

Sea $T_{pred}$ el multiconjunto de tokens de la predicción y $T_{ref}$ el de la referencia
(un *multiconjunto* cuenta repeticiones: por eso el código usa `Counter` y no `set`).
Con $|T_{pred} \cap T_{ref}|$ = número de tokens en común (contando multiplicidades):

$$P = \frac{|T_{pred} \cap T_{ref}|}{|T_{pred}|}
\qquad
R = \frac{|T_{pred} \cap T_{ref}|}{|T_{ref}|}$$

$$F_1 = \frac{2 \cdot P \cdot R}{P + R}$$

**Intuición de cada término**:
- **Precision** $P$: castiga la *verbosidad* — si el modelo agrega palabras que no están en la referencia, $P$ baja
- **Recall** $R$: castiga la *omisión* — si el modelo olvida palabras de la referencia, $R$ baja
- **$F_1$** es la **media armónica** de ambas. Se usa la armónica (y no la aritmética) porque
  penaliza fuertemente el desbalance: si $P=1.0$ pero $R=0.1$, la media aritmética daría un
  engañoso $0.55$, mientras que $F_1 = \frac{2(1.0)(0.1)}{1.1} \approx 0.18$

**Limitación estructural**: es una métrica de **bolsa de palabras** (*bag of words*) — ignora
el orden. "the dog bit the man" y "the man bit the dog" obtienen $F_1 = 1.0$ entre sí.


In [ ]:
def token_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_common = sum(common.values())

    if num_common == 0:
        return 0.0

    precision = num_common / len(pred_tokens)
    recall = num_common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

reference_long = "The capital of France is Paris"

print(f"Reference: '{reference_long}'\n")
for a in answers:
    print(f"F1 = {token_f1(a, reference_long):.2f}  ←  '{a}'")

print(f"\n⚠️ F1 = {token_f1(wrong_answer, reference_long):.2f}  ←  '{wrong_answer}' (¡INCORRECTA!)")

Reference: 'The capital of France is Paris'

F1 = 0.33  ←  'Paris'
F1 = 1.00  ←  'The capital of France is Paris.'
F1 = 0.59  ←  'Paris is the capital and largest city of France, located on the Seine river.'

⚠️ F1 = 0.80  ←  'The capital of France is Lyon.' (¡INCORRECTA!)


**🚨 Problema grave**: la respuesta **incorrecta** ("Lyon") obtiene un F1 altísimo,
porque comparte casi todas las palabras con la referencia... excepto la única que importa.

Las métricas léxicas miden *parecido superficial*, no *corrección*.

## 2.3 BLEU y ROUGE

Son las métricas léxicas clásicas de traducción (BLEU) y resumen (ROUGE).
En lugar de palabras sueltas, comparan **n-gramas** (secuencias de 1, 2, 3, 4 palabras).

- **BLEU**: orientado a *precision* de n-gramas (¿lo que generé está en la referencia?)
- **ROUGE**: orientado a *recall* (¿lo que dice la referencia está en mi resumen?)
  - ROUGE-1: unigramas, ROUGE-2: bigramas, ROUGE-L: subsecuencia común más larga

Usamos la librería `evaluate` de HuggingFace:

### 📐 Fundamento matemático: BLEU

BLEU (*Bilingual Evaluation Understudy*, Papineni et al. 2002) combina la **precision
modificada de n-gramas** para varios tamaños de n-grama:

$$p_n = \frac{\sum_{\text{ngram} \in \hat{y}} \text{Count}_{clip}(\text{ngram})}{\sum_{\text{ngram} \in \hat{y}} \text{Count}(\text{ngram})}$$

donde $\text{Count}_{clip}$ **recorta** el conteo de cada n-grama al máximo de veces que aparece
en la referencia (evita que repetir "the the the the" infle la precision).

El score final agrega los $p_n$ con media geométrica (típicamente $N=4$, pesos $w_n = 1/4$)
y multiplica por una **penalización por brevedad** ($BP$):

$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)
\qquad
BP = \begin{cases} 1 & \text{si } c > r \\ e^{\,1 - r/c} & \text{si } c \leq r \end{cases}$$

con $c$ = longitud de la predicción y $r$ = longitud de la referencia.

**¿Por qué $BP$?** BLEU es una métrica de *precision*: sin la penalización, una predicción
de una sola palabra correcta tendría precision perfecta. $BP$ castiga respuestas más cortas
que la referencia.

### 📐 Fundamento matemático: ROUGE

ROUGE (*Recall-Oriented Understudy for Gisting Evaluation*, Lin 2004) invierte la perspectiva —
mide cuánto de la **referencia** fue capturado (recall):

$$\text{ROUGE-N} = \frac{\sum_{\text{ngram}_n \in\, y} \text{Count}_{match}(\text{ngram}_n)}{\sum_{\text{ngram}_n \in\, y} \text{Count}(\text{ngram}_n)}$$

- **ROUGE-1**: unigramas (palabras sueltas) — mide cobertura de contenido
- **ROUGE-2**: bigramas — mide algo de fluidez/orden local

**ROUGE-L** usa la **subsecuencia común más larga** (LCS), que respeta el orden de las
palabras sin exigir que sean contiguas. Con $m = |y|$, $n = |\hat{y}|$:

$$R_{lcs} = \frac{\text{LCS}(\hat{y}, y)}{m} \qquad P_{lcs} = \frac{\text{LCS}(\hat{y}, y)}{n} \qquad F_{lcs} = \frac{(1+\beta^2)\, R_{lcs}\, P_{lcs}}{R_{lcs} + \beta^2 P_{lcs}}$$

**En resumen**: BLEU pregunta *"¿lo que generaste está en la referencia?"* (precision, traducción);
ROUGE pregunta *"¿lo que dice la referencia está en tu resumen?"* (recall, resúmenes).
Las implementaciones modernas (como la de `evaluate`) reportan la variante $F$ de ROUGE.


In [ ]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

prediction = "The cat sat on the mat"
reference_text = "A cat was sitting on the mat"

bleu_score = bleu.compute(predictions=[prediction], references=[[reference_text]])
rouge_score = rouge.compute(predictions=[prediction], references=[reference_text])

print(f"Prediction: '{prediction}'")
print(f"Reference:  '{reference_text}'\n")
print(f"BLEU:    {bleu_score['bleu']:.3f}")
print(f"ROUGE-1: {rouge_score['rouge1']:.3f}")
print(f"ROUGE-2: {rouge_score['rouge2']:.3f}")
print(f"ROUGE-L: {rouge_score['rougeL']:.3f}")

Prediction: 'The cat sat on the mat'
Reference:  'A cat was sitting on the mat'

BLEU:    0.000
ROUGE-1: 0.615
ROUGE-2: 0.364
ROUGE-L: 0.615


## 2.4 La gran limitación: paráfrasis

Demostremos el punto clave de esta clase. Dos oraciones que significan
**lo mismo** con palabras distintas, y dos que significan **lo contrario**
con casi las mismas palabras:

In [ ]:
cases = [
    # (predicción, referencia, ¿significan lo mismo?)
    ("The physician examined the patient", "The doctor checked the sick person", "✅ MISMO significado"),
    ("The stock market rose sharply today", "The stock market fell sharply today", "❌ significado OPUESTO"),
]

print(f"{'ROUGE-L':>8} | {'Semántica':<22} | Ejemplo")
print("-" * 90)
for pred, ref, meaning in cases:
    r = rouge.compute(predictions=[pred], references=[ref])
    print(f"{r['rougeL']:>8.2f} | {meaning:<22} | '{pred}' vs '{ref}'")

 ROUGE-L | Semántica              | Ejemplo
------------------------------------------------------------------------------------------
    0.36 | ✅ MISMO significado    | 'The physician examined the patient' vs 'The doctor checked the sick person'
    0.83 | ❌ significado OPUESTO  | 'The stock market rose sharply today' vs 'The stock market fell sharply today'


**Conclusión de la Parte 2**:

| Caso | Significado | ROUGE |
|------|------------|-------|
| Paráfrasis | Igual | **Bajo** 📉 |
| Una palabra cambiada (rose→fell) | Opuesto | **Alto** 📈 |

Las métricas léxicas se equivocan **exactamente al revés** de lo que necesitamos.
Siguen siendo útiles (baratas, deterministas, buenas para tareas cerradas),
pero para respuestas abiertas necesitamos algo más.

---
# PARTE 3: Métricas Semánticas (comparación de significado)

En vez de comparar palabras, comparamos **embeddings**: vectores que capturan
el significado del texto (los mismos que usamos para el vector store del RAG).

La métrica: **similitud coseno** entre el embedding de la predicción y el de la referencia.

### 📐 Fundamento matemático

Un modelo de embeddings es una función $E: \text{texto} \rightarrow \mathbb{R}^d$ que mapea
texto a un vector de $d$ dimensiones (para `all-MiniLM-L6-v2`, $d = 384$), entrenada para que
textos con significado similar queden **cerca** en el espacio vectorial.

La cercanía se mide con **similitud coseno** — el coseno del ángulo entre los dos vectores:

$$\text{sim}(\mathbf{u}, \mathbf{v}) = \cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|} = \frac{\sum_{j=1}^{d} u_j\, v_j}{\sqrt{\sum_j u_j^2}\, \sqrt{\sum_j v_j^2}}$$

**Propiedades**:
- Rango teórico $[-1, 1]$: $1$ = misma dirección, $0$ = ortogonales, $-1$ = opuestos.
  En la práctica, con embeddings de oraciones los valores suelen caer en $[0, 1]$
- Mide **dirección**, no magnitud: es invariante a la longitud del vector
- Si los embeddings están **normalizados** ($\|\mathbf{u}\| = \|\mathbf{v}\| = 1$, como hacemos
  con `normalize_embeddings=True`), la fórmula se reduce al producto punto:
  $\text{sim}(\mathbf{u},\mathbf{v}) = \mathbf{u} \cdot \mathbf{v}$ — por eso el código usa `np.dot`

> 💡 Es exactamente la misma matemática que usa el **vector store** del notebook 12 para
> recuperar documentos: allí comparábamos *query vs documentos*; aquí comparamos
> *predicción vs referencia*.


In [ ]:
print("🔄 Loading embeddings model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")   # mismo modelo del notebook 12
print("✅ Embeddings ready")

def semantic_similarity(prediction, reference):
    emb = embedder.encode([prediction, reference], normalize_embeddings=True)
    return float(np.dot(emb[0], emb[1]))

🔄 Loading embeddings model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embeddings ready


In [ ]:
print(f"{'ROUGE-L':>8} | {'Semántica':>9} | Caso")
print("-" * 95)
for pred, ref, meaning in cases:
    r = rouge.compute(predictions=[pred], references=[ref])['rougeL']
    s = semantic_similarity(pred, ref)
    print(f"{r:>8.2f} | {s:>9.2f} | {meaning}: '{pred}' vs '{ref}'")

# Y nuestro caso de París
print()
for a in answers:
    print(f"Semántica = {semantic_similarity(a, 'The capital of France is Paris'):.2f}  ←  '{a}'")
print(f"Semántica = {semantic_similarity(wrong_answer, 'The capital of France is Paris'):.2f}  ←  '{wrong_answer}' ❌")

 ROUGE-L | Semántica | Caso
-----------------------------------------------------------------------------------------------
    0.36 |      0.53 | ✅ MISMO significado: 'The physician examined the patient' vs 'The doctor checked the sick person'
    0.83 |      0.79 | ❌ significado OPUESTO: 'The stock market rose sharply today' vs 'The stock market fell sharply today'

Semántica = 0.70  ←  'Paris'
Semántica = 0.99  ←  'The capital of France is Paris.'
Semántica = 0.82  ←  'Paris is the capital and largest city of France, located on the Seine river.'
Semántica = 0.75  ←  'The capital of France is Lyon.' ❌


**Observaciones**:
- La paráfrasis (doctor/physician) ahora obtiene un score **alto** ✅
- Las tres respuestas correctas de París obtienen scores altos ✅
- **Pero ojo**: "rose" vs "fell" y "Paris" vs "Lyon" todavía obtienen similitud
  relativamente alta, porque las oraciones se parecen en casi todo 🚨

👉 Los embeddings capturan *tema* muy bien, pero pueden ser insensibles a
negaciones o cambios de un solo dato crítico. **Ninguna métrica automática es perfecta** —
por eso en la Clase 4 veremos LLM-as-a-judge, y por eso siempre se combinan varias señales.

> 💡 **Nota**: BERTScore es otra métrica semántica popular (compara embeddings
> token a token en vez de la oración completa). Puedes probarla con
> `evaluate.load("bertscore")`.

---
# PARTE 4: Perplexity — ¿Qué tan "sorprendido" está el modelo?

Todas las métricas anteriores comparan una **salida contra una referencia**.
La perplexity es diferente: es una métrica **intrínseca** y **sin referencia** —
mide qué tan bien el propio modelo de lenguaje *predice* un texto, token a token.

### 📐 Fundamento matemático

Un modelo de lenguaje autorregresivo asigna probabilidad a una secuencia
$x = (x_1, x_2, \ldots, x_N)$ mediante la **regla de la cadena**:

$$p_\theta(x) = \prod_{i=1}^{N} p_\theta(x_i \mid x_{<i})$$

es decir, la probabilidad de cada token dado todo lo anterior. A partir de ahí definimos
la **entropía cruzada** promedio por token (que es exactamente la *loss* con la que se
entrenan los LLMs):

$$\mathcal{H}(x) = -\frac{1}{N}\sum_{i=1}^{N} \log p_\theta(x_i \mid x_{<i})$$

y la **perplexity** es su exponencial:

$$\text{PPL}(x) = \exp\big(\mathcal{H}(x)\big) = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log p_\theta(x_i \mid x_{<i})\right)$$

### 🎲 Interpretación intuitiva

La perplexity es el **factor de ramificación efectivo**: el número promedio de tokens entre
los que el modelo "duda" en cada paso.

- $\text{PPL} = 1$ → el modelo predice cada token con certeza total (probabilidad 1)
- $\text{PPL} = 50$ → en promedio, es como si el modelo eligiera *uniformemente entre 50 opciones* en cada paso
- Un modelo que eligiera al azar sobre un vocabulario de $V$ tokens tendría $\text{PPL} = V$ (¡decenas de miles!)

👉 **Menor perplexity = el texto es más "esperable" para el modelo.**

### ⚠️ Antes de medir, tres advertencias

1. **Depende del tokenizador**: no se pueden comparar perplexities entre modelos con
   vocabularios distintos (GPT-2 vs LLaMA, por ejemplo). Solo comparar mismo modelo /
   misma tokenización.
2. **Requiere un modelo causal (autorregresivo)**: nuestro `flan-t5` es *seq2seq*
   (encoder-decoder), donde la definición estándar no aplica directamente. Por eso
   usaremos **GPT-2** para esta sección.
3. **Fluidez ≠ verdad**: un texto falso pero bien escrito tiene perplexity baja.
   Lo demostraremos.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer as CausalTokenizer

print("🔄 Loading GPT-2 (causal LM) for perplexity...")
ppl_tokenizer = CausalTokenizer.from_pretrained("gpt2")
ppl_model = AutoModelForCausalLM.from_pretrained("gpt2")
ppl_model.eval()
print("✅ GPT-2 ready")


def perplexity(text):
    """
    PPL(x) = exp( -1/N * sum_i log p(x_i | x_<i) )

    Al pasar labels=input_ids, HuggingFace calcula internamente la
    entropía cruzada promedio por token (out.loss = H). Solo falta exp().
    """
    inputs = ppl_tokenizer(text, return_tensors="pt") # hola : 1, que : 4 -> 1,2,5,100, , attns
    with torch.no_grad():
        out = ppl_model(inputs.input_ids, labels=inputs.input_ids)
    return torch.exp(out.loss).item()   # PPL = exp(H)

print("✅ perplexity() defined")

🔄 Loading GPT-2 (causal LM) for perplexity...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ GPT-2 ready
✅ perplexity() defined


## Experimento 1: fluidez y coherencia

Comparemos textos con las **mismas palabras** pero distinto orden, y texto sin sentido:

In [ ]:
texts = [
    ("Texto fluido",       "The cat sat quietly on the warm mat near the window."),
    ("Mismas palabras, desordenadas", "Mat the near quietly warm sat the on cat window the."),
    ("Texto técnico coherente", "Machine learning models are trained on large datasets to make predictions."),
    ("Pseudo-texto sin sentido", "Colorless green ideas sleep furiously under the seven of blue."),
    ("Caracteres aleatorios", "xq zzv brfk lmnp qwrt jkfd oplm"),
]

print(f"{'PPL':>10} | Texto")
print("-" * 80)
for label, t in texts:
    print(f"{perplexity(t):>10.1f} | [{label}] '{t}'")

       PPL | Texto
--------------------------------------------------------------------------------


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


      65.2 | [Texto fluido] 'The cat sat quietly on the warm mat near the window.'
    5196.6 | [Mismas palabras, desordenadas] 'Mat the near quietly warm sat the on cat window the.'
      44.5 | [Texto técnico coherente] 'Machine learning models are trained on large datasets to make predictions.'
    1376.8 | [Pseudo-texto sin sentido] 'Colorless green ideas sleep furiously under the seven of blue.'
     201.8 | [Caracteres aleatorios] 'xq zzv brfk lmnp qwrt jkfd oplm'


**Observaciones**:
- El mismo vocabulario **desordenado** dispara la perplexity: el modelo captura sintaxis y orden
  (algo que Token F1 y ROUGE-1 no pueden ver)
- El texto aleatorio tiene perplexity altísima

## Experimento 2: fluidez ≠ verdad

Volvamos al ejemplo central de la clase. ¿La perplexity detecta que "Lyon" es incorrecto?

In [ ]:
factual = "The capital of France is Paris."
false_but_fluent = "The capital of France is Lyon."

print(f"PPL = {perplexity(factual):>8.1f}  |  '{factual}' ✅ verdadero")
print(f"PPL = {perplexity(false_but_fluent):>8.1f}  |  '{false_but_fluent}' ❌ falso")

print("""
🚨 Ambas oraciones son perfectamente fluidas → perplexities similares.
   La perplexity mide qué tan ESPERABLE es un texto, no qué tan VERDADERO.
""")

PPL =     44.6  |  'The capital of France is Paris.' ✅ verdadero
PPL =     80.9  |  'The capital of France is Lyon.' ❌ falso

🚨 Ambas oraciones son perfectamente fluidas → perplexities similares.
   La perplexity mide qué tan ESPERABLE es un texto, no qué tan VERDADERO.



## ¿Para qué sirve entonces la perplexity?

| Uso | Ejemplo |
|-----|---------|
| **Comparar modelos base** | ¿El modelo A modela mejor el inglés que el B? (mismo tokenizador/datos de test) |
| **Medir adaptación a dominio** | Tras fine-tuning en textos médicos, ¿bajó la PPL sobre textos médicos? |
| **Filtrar calidad de datos** | Descartar documentos con PPL altísima (basura, texto corrupto) del corpus de entrenamiento |
| **Detección de anomalías** | Inputs muy raros (posibles ataques, spam, idioma inesperado) tienen PPL alta |
| **Monitorear entrenamiento** | La loss de entrenamiento ES la entropía cruzada: $\text{PPL} = e^{\text{loss}}$ |

Y sus **límites**, que ya demostramos:
- No mide veracidad, utilidad ni seguimiento de instrucciones
- No es comparable entre tokenizadores distintos
- Un modelo alineado con RLHF puede tener PPL "peor" sobre texto de internet y aun así ser mucho más útil

> 💡 También existe `evaluate.load("perplexity")` para calcularla en batch sobre datasets.


---
# PARTE 5: Benchmarks — Evaluando sobre MMLU

Los **benchmarks** son datasets estandarizados que permiten comparar modelos entre sí.
Algunos famosos:

| Benchmark | Qué mide |
|-----------|----------|
| **MMLU** | Conocimiento general (57 materias, opción múltiple) |
| **HellaSwag** | Sentido común |
| **HumanEval** | Generación de código |
| **TruthfulQA** | Tendencia a repetir falsedades comunes |
| **MT-Bench / Chatbot Arena** | Calidad conversacional (con jueces) |

Vamos a evaluar nuestro `flan-t5-base` sobre una muestra de **MMLU**.
La ventaja de la opción múltiple: la evaluación vuelve a ser un problema
de clasificación → podemos usar **accuracy** sin ambigüedad.

### 📐 Fundamento matemático

Con opción múltiple, la evaluación vuelve a ser clasificación pura:

$$\text{Accuracy} = \frac{1}{N}\sum_{i=1}^{N} \mathbb{1}\big[\hat{y}_i = y_i\big] = \frac{\text{aciertos}}{N}$$

Con $k$ opciones por pregunta, el **baseline aleatorio** es $\frac{1}{k}$ (en MMLU, $k=4$
→ $25\%$): cualquier score debe interpretarse relativo a ese piso.

**Nota estadística**: con muestras pequeñas hay mucha varianza. El error estándar de una
proporción es $SE = \sqrt{\frac{p(1-p)}{N}}$ — con $N=30$ y $p=0.5$, $SE \approx 9\%$.
Por eso los benchmarks serios usan cientos o miles de preguntas.


In [ ]:
from datasets import load_dataset

print("🔄 Loading MMLU sample...")
mmlu = load_dataset("cais/mmlu", "high_school_geography", split="test")
print(f"✅ {len(mmlu)} questions loaded")
print(f"\nEjemplo:")
example = mmlu[0]
print(f"  Question: {example['question']}")
for letter, choice in zip("ABCD", example['choices']):
    print(f"    {letter}) {choice}")
print(f"  Answer: {'ABCD'[example['answer']]}")

🔄 Loading MMLU sample...


README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

high_school_geography/test-00000-of-0000(…): reconstructing file:   0%|          |  0.00B / 28.2kB            

high_school_geography/test-00000-of-0000(…): downloading bytes:           |  0.00B            

...

high_school_geography/dev-00000-of-00001(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/198 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/22 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

✅ 198 questions loaded

Ejemplo:
  Question: The main factor preventing subsistence economies from advancing economically is the lack of
    A) a currency.
    B) a well-connected transportation infrastructure.
    C) government activity.
    D) a banking service.
  Answer: B


In [ ]:
def format_mmlu_prompt(item):
    """Convierte una pregunta MMLU en un prompt de opción múltiple"""
    options = "\n".join(
        f"{letter}) {choice}"
        for letter, choice in zip("ABCD", item["choices"])
    )
    return (
        f"Answer the following multiple choice question with A, B, C or D.\n\n"
        f"Question: {item['question']}\n{options}\n\nAnswer:"
    )

def extract_letter(generated_text):
    """Extrae la letra de la respuesta generada"""
    match = re.search(r"[ABCD]", generated_text.strip().upper())
    return match.group(0) if match else None

# Evaluar sobre N preguntas (sube este número si tienes GPU)
N = 30
correct = 0
results = []

print(f"🔄 Evaluating on {N} questions...\n")
for i in range(N):
    item = mmlu[i]
    prompt = format_mmlu_prompt(item)

    output = llm(prompt, max_length=10)[0]["generated_text"]

    predicted = extract_letter(output)
    expected = "ABCD"[item["answer"]]
    is_correct = predicted == expected

    correct += is_correct
    results.append((predicted, expected, is_correct))

    if i < 5:  # mostrar los primeros 5
        icon = "✅" if is_correct else "❌"
        print(f"{icon} Q{i+1}: predicted={predicted}, expected={expected}")

accuracy = correct / N
print(f"\n{'='*50}")
print(f"📊 ACCURACY: {accuracy:.1%} ({correct}/{N})")
print(f"{'='*50}")
print(f"\n💡 Random baseline: 25% — ¿nuestro modelo lo supera?")

🔄 Evaluating on 30 questions...

❌ Q1: predicted=A, expected=B
❌ Q2: predicted=A, expected=D
❌ Q3: predicted=A, expected=C
✅ Q4: predicted=A, expected=A
❌ Q5: predicted=A, expected=D


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



📊 ACCURACY: 20.0% (6/30)

💡 Random baseline: 25% — ¿nuestro modelo lo supera?


**Reflexión importante**: los benchmarks públicos tienen problemas conocidos:

1. **Contaminación**: si las preguntas del benchmark estaban en los datos de
   entrenamiento del modelo, el score está inflado.
2. **Saturación**: los modelos punteros ya superan 85-90% en MMLU; deja de discriminar.
3. **No miden TU caso de uso**: un modelo puede ser excelente en MMLU y pésimo
   respondiendo preguntas sobre los documentos internos de tu empresa.

👉 Por eso el punto 3 nos lleva a la parte más importante para la práctica profesional:

---
# PARTE 6: Construcción de un Golden Dataset propio

Un **golden dataset** (o *golden set*) es TU benchmark: pares de
(entrada, salida esperada) curados manualmente para TU aplicación.

Reglas de oro:
1. **Representativo**: debe cubrir los tipos de preguntas reales de tus usuarios
2. **Incluir casos difíciles**: ambiguos, fuera de dominio, adversariales
3. **Empezar pequeño**: 20-50 ejemplos bien curados > 1000 mal hechos
4. **Versionarlo**: el golden set evoluciona con el producto (lo veremos en LLMOps)

Construyamos uno para el sistema RAG del notebook 12 (documentos sobre RAG, LLMs,
vector DBs, BM25 y LangChain):

In [ ]:
golden_dataset = [
    # Casos normales
    {
        "question": "What are the two main stages of RAG?",
        "reference": "First retrieval of relevant documents, then generation of a response using them as context.",
        "type": "normal"
    },
    {
        "question": "Name three vector databases",
        "reference": "Chroma, Pinecone, Weaviate, FAISS and Qdrant are vector databases.",
        "type": "normal"
    },
    {
        "question": "What is BM25 good at?",
        "reference": "BM25 is good at finding exact word matches like specific terms, names, or technical jargon.",
        "type": "normal"
    },
    {
        "question": "Why is RAG useful for LLMs?",
        "reference": "RAG gives LLMs access to up-to-date information without retraining, reduces hallucinations, and provides sources.",
        "type": "normal"
    },
    # Caso difícil: requiere combinar dos documentos
    {
        "question": "How does hybrid search combine BM25 and embeddings?",
        "reference": "Hybrid search combines semantic search using embeddings for meaning with BM25 for exact keyword matches.",
        "type": "multi-doc"
    },
    # Caso fuera de dominio: el sistema NO debería inventar
    {
        "question": "What is the population of Tokyo?",
        "reference": "The documents do not contain information about Tokyo.",
        "type": "out-of-domain"
    },
]

print(f"✅ Golden dataset: {len(golden_dataset)} examples")
for item in golden_dataset:
    print(f"   [{item['type']:<13}] {item['question']}")

✅ Golden dataset: 6 examples
   [normal       ] What are the two main stages of RAG?
   [normal       ] Name three vector databases
   [normal       ] What is BM25 good at?
   [normal       ] Why is RAG useful for LLMs?
   [multi-doc    ] How does hybrid search combine BM25 and embeddings?
   [out-of-domain] What is the population of Tokyo?


## Evaluación automática contra el golden set

Ahora ejecutamos nuestro LLM contra el golden set y calculamos las métricas
de esta clase en batch. Este patrón — *dataset → predicciones → métricas → tabla* —
es el corazón de todo lo que haremos en el resto del curso.

In [ ]:
# Contexto simulado (en la Clase 2 usaremos el retriever real del notebook 12)
context = """RAG works in two stages: first retrieving relevant documents, then generating
a response with an LLM using them as context. RAG allows access to up-to-date information
without retraining and reduces hallucinations. Common vector databases include Chroma,
Pinecone, Weaviate, FAISS, and Qdrant. BM25 finds exact word matches and is good with
specific terms, names or jargon. Hybrid search combines semantic embeddings with BM25."""

def run_evaluation(dataset):
    rows = []
    for item in dataset:
        prompt = f"Answer based on the context.\n\nContext: {context}\n\nQuestion: {item['question']}\n\nAnswer:"
        prediction = llm(prompt)[0]["generated_text"]

        rows.append({
            "type": item["type"],
            "question": item["question"],
            "prediction": prediction,
            "f1": token_f1(prediction, item["reference"]),
            "rougeL": rouge.compute(predictions=[prediction], references=[item["reference"]])["rougeL"],
            "semantic": semantic_similarity(prediction, item["reference"]),
        })
    return rows

print("🔄 Running evaluation...\n")
rows = run_evaluation(golden_dataset)

print(f"{'Type':<13} | {'F1':>5} | {'ROUGE-L':>7} | {'Semantic':>8} | Prediction")
print("-" * 100)
for r in rows:
    print(f"{r['type']:<13} | {r['f1']:>5.2f} | {r['rougeL']:>7.2f} | {r['semantic']:>8.2f} | {r['prediction'][:45]}")

print(f"\n{'='*50}")
print(f"📊 AVERAGES")
print(f"{'='*50}")
print(f"F1:       {np.mean([r['f1'] for r in rows]):.3f}")
print(f"ROUGE-L:  {np.mean([r['rougeL'] for r in rows]):.3f}")
print(f"Semantic: {np.mean([r['semantic'] for r in rows]):.3f}")

🔄 Running evaluation...

Type          |    F1 | ROUGE-L | Semantic | Prediction
----------------------------------------------------------------------------------------------------
normal        |  0.23 |    0.21 |     0.41 | Answer based on the context.

Context: RAG wo
normal        |  0.20 |    0.19 |     0.49 | Answer based on the context.

Context: RAG wo
normal        |  0.28 |    0.20 |     0.64 | Answer based on the context.

Context: RAG wo
normal        |  0.25 |    0.26 |     0.71 | Answer based on the context.

Context: RAG wo
multi-doc     |  0.24 |    0.15 |     0.75 | Answer based on the context.

Context: RAG wo
out-of-domain |  0.08 |    0.09 |     0.33 | Answer based on the context.

Context: RAG wo

📊 AVERAGES
F1:       0.213
ROUGE-L:  0.182
Semantic: 0.556


**Preguntas para discutir en clase**:
- ¿En qué tipo de casos las métricas coinciden? ¿En cuáles divergen?
- ¿Qué pasó con el caso *out-of-domain*? ¿La métrica premió o castigó lo correcto?
- ¿Un promedio global oculta problemas? (pista: mira los scores **por tipo de caso**)

---
# PARTE 7: Generación Sintética de Datos de Evaluación

Curar golden sets a mano es caro. Un truco muy usado en la industria:
**usar un LLM para generar preguntas** a partir de tus documentos.

Pipeline típico:
1. Tomar un chunk de documento
2. Pedirle al LLM que genere una pregunta que ese chunk responde
3. El chunk se convierte en el "contexto golden" y sirve para derivar la respuesta esperada
4. **Un humano revisa y filtra** (¡esto no es opcional!)

In [ ]:
# Chunks de los documentos del notebook 12
source_chunks = [
    "Embeddings are typically 384 to 1536 dimensions. Models like Sentence-BERT or OpenAI's embedding models convert text into these vectors.",
    "LangGraph extends LangChain with state graphs for building complex workflows with cycles, conditional logic, and state management.",
    "LLMs have limitations: their knowledge is frozen at training time, they can hallucinate false information, and they cannot access private or recent data.",
]

print("🔄 Generating synthetic questions...\n")
synthetic_dataset = []

for chunk in source_chunks:
    prompt = f"Generate one question that can be answered using only this text:\n\n{chunk}\n\nQuestion:"
    question = llm(prompt)[0]["generated_text"]

    synthetic_dataset.append({
        "question": question,
        "golden_context": chunk,
    })
    print(f"📄 Chunk: {chunk[:60]}...")
    print(f"❓ Generated question: {question}\n")

print(f"✅ {len(synthetic_dataset)} synthetic examples generated")

🔄 Generating synthetic questions...

📄 Chunk: Embeddings are typically 384 to 1536 dimensions. Models like...
❓ Generated question: Generate one question that can be answered using only this text:

Embeddings are typically 384 to 1536 dimensions. Models like Sentence-BERT or OpenAI's embedding models convert text into these vectors.

Question:

📄 Chunk: LangGraph extends LangChain with state graphs for building c...
❓ Generated question: Generate one question that can be answered using only this text:

LangGraph extends LangChain with state graphs for building complex workflows with cycles, conditional logic, and state management.

Question:

📄 Chunk: LLMs have limitations: their knowledge is frozen at training...
❓ Generated question: Generate one question that can be answered using only this text:

LLMs have limitations: their knowledge is frozen at training time, they can hallucinate false information, and they cannot access private or recent data.

Question:

✅ 3 synthetic examples

**⚠️ Advertencias sobre datos sintéticos**:
- Las preguntas generadas tienden a ser *demasiado fáciles* (copian palabras del chunk)
- Pueden heredar sesgos y errores del LLM generador
- Con un modelo pequeño como flan-t5-base la calidad es limitada — en producción
  se usa un modelo potente para generar y **siempre** hay revisión humana
- Frameworks como RAGAS (Clase 3) automatizan y mejoran este pipeline

---
# PARTE 8: Evaluación Offline vs Online

Todo lo que hicimos hoy es **evaluación offline**: medimos contra un dataset fijo,
antes de desplegar. Es la mitad de la historia.

| | **Offline** | **Online** |
|---|---|---|
| **Cuándo** | Antes de desplegar | En producción |
| **Datos** | Golden dataset | Tráfico real de usuarios |
| **Señales** | Métricas automáticas, jueces | Feedback (👍/👎), tasa de reintento, abandono |
| **Sirve para** | Comparar modelos/prompts, regression testing | Detectar drift, problemas reales |
| **Riesgo** | El dataset no representa la realidad | Los usuarios sufren los errores |

**El ciclo virtuoso** (lo construiremos completo durante el curso):

```
   Golden dataset ──► Evaluación offline ──► Deploy
        ▲                                      │
        │                                      ▼
   Casos nuevos ◄── Análisis de fallos ◄── Monitoreo online
```

Los fallos detectados en producción se convierten en nuevos casos del golden set.
Esto conecta la evaluación (Bloque 1) con la observabilidad (Bloque 2) y LLMOps (Bloque 3).

---
# Ejercicios Propuestos

1. **Métricas**: agrega BERTScore (`evaluate.load("bertscore")`) a la tabla de la Parte 6
   y compárala con la similitud coseno. ¿Cambia alguna conclusión?

2. **Adversarial**: escribe 3 pares (predicción, referencia) donde la similitud
   semántica falle: score alto con significados opuestos. Pista: negaciones,
   números, nombres propios.

3. **Benchmark**: cambia la materia de MMLU (`"high_school_geography"` →
   `"elementary_mathematics"`, `"computer_security"`, etc.) y compara el accuracy.
   ¿En qué materias es mejor el modelo? ¿Por qué crees que ocurre?

4. **Golden set**: agrega 5 casos nuevos al golden dataset, incluyendo al menos
   un caso *adversarial* (pregunta con premisa falsa, ej. "Why is FAISS a relational
   database?") y evalúa cómo se comporta el sistema.

5. **Perplexity**: calcula la perplexity de las respuestas generadas por el LLM en la
   Parte 6 (golden dataset). ¿Las respuestas con mejor similitud semántica tienen
   también menor perplexity? ¿Debería haber relación?

6. **Reflexión**: si tuvieras que elegir UNA sola métrica para un pipeline de CI
   que corre en cada cambio de prompt, ¿cuál elegirías y por qué? ¿Qué riesgo aceptas?


---
# Conclusión

### Lo que aprendimos

| Métrica | Mide | Fortaleza | Debilidad |
|---------|------|-----------|-----------|
| **Exact Match** | Igualdad exacta | Sin ambigüedad | Castiga paráfrasis |
| **Token F1** | Solapamiento de palabras | Simple, rápida | Premia parecido superficial |
| **BLEU / ROUGE** | N-gramas | Estándar histórico | Falla con paráfrasis |
| **Similitud semántica** | Significado (embeddings) | Robusta a paráfrasis | Insensible a negaciones/datos puntuales |
| **Perplexity** | Fluidez / predictibilidad (intrínseca, sin referencia) | No requiere referencia; detecta texto anómalo | No mide veracidad; no comparable entre tokenizadores |
| **Accuracy (benchmarks)** | Opción múltiple | Objetiva, comparable | Contaminación, no mide TU caso |

### Ideas clave
1. **No existe LA métrica** — se combinan varias señales
2. **Tu golden dataset vale más que cualquier benchmark público**
3. Los promedios ocultan problemas: analiza **por tipo de caso**
4. Offline y online se retroalimentan